In [2]:
import sys
print(sys.executable)

c:\Users\Lenne\Documents\School\Master\Sem2\Data_Engineering\DE-project\venv\Scripts\python.exe


In [3]:
# If needed, install dependencies first:
# !pip install yfinance pandas pyarrow

import yfinance as yf
import pandas as pd

# 11 SPDR Sector ETFs
# Note: XLRE only starts in 2015, so it will not have data back to 2000.
sector_etfs = {
    "Energy": "XLE",
    "Technology": "XLK",          # use "QQQ" instead if your assignment prefers it
    "Financials": "XLF",
    "Consumer_Staples": "XLP",
    "Consumer_Discretionary": "XLY",
    "Industrials": "XLI",
    "Healthcare": "XLV",
    "Real_Estate": "XLRE",
    "Materials": "XLB",
    "Utilities": "XLU",
    "Communication_Services": "XLC"
}

tickers = list(sector_etfs.values())

# Download monthly data
prices = yf.download(
    tickers=tickers,
    start="2000-01-01",
    interval="1mo",
    auto_adjust=True,
    progress=False
)

# Get monthly closing prices
monthly_close = prices["Close"].copy()

# Rename ticker columns to sector names
ticker_to_sector = {ticker: sector for sector, ticker in sector_etfs.items()}
monthly_close = monthly_close.rename(columns=ticker_to_sector)

# Compute monthly percentage returns
monthly_returns = monthly_close.pct_change() * 100

# Reset index so DATE is a column
monthly_returns = monthly_returns.reset_index()

# Format date column like FRED data: DATE = YYYY-MM-DD
monthly_returns = monthly_returns.rename(columns={"Date": "DATE"})
monthly_returns["DATE"] = pd.to_datetime(monthly_returns["DATE"]).dt.strftime("%Y-%m-%d")

# Optional: drop the first row because returns are NaN
monthly_returns = monthly_returns.dropna(how="all", subset=list(sector_etfs.keys()))

# Save as Parquet
monthly_returns.to_parquet("spdr_sector_monthly_returns.parquet", index=False)

monthly_returns.head()


Ticker,DATE,Materials,Communication_Services,Energy,Financials,Industrials,Technology,Consumer_Staples,Real_Estate,Utilities,Healthcare,Consumer_Discretionary
1,2000-02-01,-10.046879,NaN,-4.233397,-10.704589,-5.517274,10.506765,-11.701429,NaN,-12.153762,-6.538580,-5.571483
2,2000-03-01,9.828679,NaN,12.066932,17.829983,13.686127,8.389248,3.427231,NaN,10.102932,8.994941,13.929450
3,2000-04-01,-3.092626,NaN,-1.162496,1.336368,1.653935,-9.184733,5.639967,NaN,7.340071,-1.171659,-1.891207
4,2000-05-01,-3.155670,NaN,11.742396,2.232115,-0.421933,-10.397711,7.132880,NaN,-0.219421,-2.680407,-5.398039
5,2000-06-01,-8.978965,NaN,-5.956394,-5.177724,-4.131385,9.955623,5.417757,NaN,-4.123155,0.211855,-5.590800
